# FASE 5: Integración y Sistema Completo

**Autor:** Jaime Meléndez Zambrano

## Objetivo
Integrar las tres técnicas (clasificación supervisada, clustering no
supervisado y generación con Markov) en un único sistema
(`SistemaNLPCompleto`, en `src/sistema_completo.py`), y ponerlo a prueba
con los **15 "relatos sin etiqueta"** que se separaron en la Fase 0 (5 por
categoría): textos reales de la encuesta que el sistema nunca vio durante
el entrenamiento.

In [1]:
import os
import sys
import random
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../src'))
from sistema_completo import SistemaNLPCompleto

random.seed(42)
pd.set_option('display.max_colwidth', 90)


## Cargar el corpus y entrenar las tres piezas del sistema

In [2]:
sistema = SistemaNLPCompleto()
sistema.cargar_corpus('../data/corpus_etiquetado')


Corpus cargado: 843 textos
Categorías: ['desagradable', 'mixto', 'placentero']


(['a veces me siento agotada porque me acuesto tarde estudiando y en el colegio me da sueño, pero en la casa trato de organizar mi tiempo libre.',
  'me siento alegre en el colegio por estar con mis amigos, pero en casa a veces me aburro un poco si no tengo planes o si hay mucho por hacer.',
  'a veces frustrado porque siento que pasamos casi todo el día enfocados en el colegio y en la casa el tiempo libre se va rápido en deberes.',
  'opino que a veces acumulan muchas tareas para la misma fecha y al momento de hacerlas se nos hace muy difícil rendir en todas.',
  'me estreso con ese poco de tareas y talleres que dejan seguidos, especialmente cuando se juntan con los exámenes parciales.',
  'opino que es mejor hacer las tareas el mismo día que las ponen porque si acumulamos todo se nos hace más difícil entregar a tiempo.',
  'mucha tarea ya que en momentos tenemos trabajos acumulados y al mismo tiempo proyectos, exposiciones y evaluaciones, es mucho compromiso.',
  'siento que dejan mu

In [3]:
resultados_clasificacion = sistema.entrenar_clasificadores(verbose=False)
for nombre, r in resultados_clasificacion.items():
    print(f"{nombre}: accuracy={r['accuracy']:.3f}")


Entrenamiento: 674 textos
Prueba: 169 textos


Modelos entrenados: ['Naive Bayes', 'Logistic Regression', 'SVM']
Naive Bayes: accuracy=0.775
Logistic Regression: accuracy=0.787
SVM: accuracy=0.799


In [4]:
analisis_clusters = sistema.hacer_clustering(n_clusters=3)
for cid, info in analisis_clusters.items():
    print(f"Cluster {cid}: {info['num_textos']} textos, mayoría={info['categoria_principal']}")


3 clusters creados
  Cluster 0: 56 textos
  Cluster 1: 354 textos
  Cluster 2: 433 textos
Cluster 0: 56 textos, mayoría=desagradable
Cluster 1: 354 textos, mayoría=desagradable
Cluster 2: 433 textos, mayoría=placentero


In [5]:
generadores = sistema.entrenar_generadores()


Entrenado con 325 textos (orden 1)
Estados distintos: 768
Entrenado con 325 textos (orden 2)
Estados distintos: 2159
Entrenado con 205 textos (orden 1)
Estados distintos: 799
Entrenado con 205 textos (orden 2)
Estados distintos: 2117
Entrenado con 313 textos (orden 1)
Estados distintos: 749
Entrenado con 313 textos (orden 2)
Estados distintos: 2145
Generadores entrenados para 3 categorías


## Sistema en acción: predecir y generar a partir de un prompt libre

In [6]:
ejemplo = sistema.predecir_y_generar("Era un día pesado en el colegio y")
print(f"Prompt: \"{ejemplo['prompt_original']}\"")
print(f"Categoría predicha: {ejemplo['categoria_predicha']}")
print(f"Texto generado:")
print(f"  {ejemplo['texto_generado']}")


Prompt: "Era un día pesado en el colegio y"
Categoría predicha: desagradable
Texto generado:
  colegio y llego muy cansado a casa, por lo que casi no tengo. pa la casa trato de organizar mi tiempo libre. Entre las tareas son demasiado extensas y no dejar materias libre es muy valioso y casi no puedo


## Predicción de los 15 "relatos sin etiqueta" (textos de prueba)

Estos textos fueron separados en la Fase 0 y su etiqueta real se ocultó
del sistema durante todo el entrenamiento (5 por categoría). Aquí se revela
esa etiqueta real únicamente para evaluar qué tan bien predice el
sistema.

In [7]:
holdout = pd.read_csv('../data/relatos_sin_etiqueta.csv')
print(f"Total de textos de prueba: {len(holdout)}")
holdout[['id', 'encuestador', 'etiqueta', 'texto']]


Total de textos de prueba: 15


,id,encuestador,etiqueta,texto
0,texto_0046,abraham,desagradable,estamos en grado décimo y las actividades son más extensas y complejas que en años ant...
1,texto_0481,jorge,desagradable,Siento que paso mucho tiempo en la escuela y me queda poco tiempo libre en casa.
2,texto_0387,jair,desagradable,"Son muchos además que son de un día para otro para realizar tareas largas, entre los e..."
3,texto_0120,eberto,desagradable,"Paso gran parte del día en la institución y normalmente llego muy cansado a casa, por ..."
4,texto_0244,elvis,desagradable,Pienso que el horario escolar es muy largo.
5,texto_0241,elvis,placentero,Creo que el tiempo libre que tengo es suficiente si me organizo.
6,texto_0796,menco,placentero,"Creo que el colegio es importante, pero el tiempo en casa también es necesario."
7,texto_0098,eberto,placentero,El horario me parece adecuado porque todavía tengo tiempo para descansar y compartir c...
8,texto_0693,marlon,placentero,Bien porque pasamos mas tiempo en casa que en clase yo digo el que tenemos es ideal pa...
9,texto_0024,abraham,placentero,"me siento cómodo en el colegio porque hay buen ambiente en el salón, y en mi casa apro..."


## Reporte completo (clasificación + cluster + generación) sobre los 15 textos de prueba

In [8]:
reporte = sistema.generar_reporte(holdout['texto'].tolist())
reporte_mostrar = reporte[[
    'texto_id', 'texto', 'clasificacion_nb', 'clasificacion_lr', 'consenso', 'cluster', 'categoria_cluster'
]].copy()
reporte_mostrar['texto'] = reporte_mostrar['texto'].str.slice(0, 60) + '...'
reporte_mostrar


,texto_id,texto,clasificacion_nb,clasificacion_lr,consenso,cluster,categoria_cluster
0,1,estamos en grado décimo y las actividades son más extensas y...,desagradable,desagradable,desagradable,1,desagradable
1,2,Siento que paso mucho tiempo en la escuela y me queda poco t...,desagradable,desagradable,desagradable,2,placentero
2,3,Son muchos además que son de un día para otro para realizar ...,desagradable,desagradable,desagradable,1,desagradable
3,4,Paso gran parte del día en la institución y normalmente lleg...,desagradable,desagradable,desagradable,2,placentero
4,5,Pienso que el horario escolar es muy largo....,placentero,placentero,placentero,1,desagradable
5,6,Creo que el tiempo libre que tengo es suficiente si me organ...,placentero,placentero,placentero,2,placentero
6,7,"Creo que el colegio es importante, pero el tiempo en casa ta...",mixto,mixto,mixto,2,placentero
7,8,El horario me parece adecuado porque todavía tengo tiempo pa...,placentero,placentero,placentero,2,placentero
8,9,Bien porque pasamos mas tiempo en casa que en clase yo digo ...,placentero,placentero,placentero,2,placentero
9,10,me siento cómodo en el colegio porque hay buen ambiente en e...,placentero,placentero,placentero,2,placentero


## Tabla final: clasificación NB/LR/Cluster vs. etiqueta real

In [9]:
tabla_final = pd.DataFrame({
    "ID": holdout['id'],
    "Encuestador": holdout['encuestador'],
    "Texto (primeras 12 palabras)": holdout['texto'].apply(lambda t: " ".join(t.split()[:12]) + "..."),
    "Etiqueta real": holdout['etiqueta'],
    "Clasificación NB": reporte['clasificacion_nb'].values,
    "Clasificación LR": reporte['clasificacion_lr'].values,
    "Consenso": reporte['consenso'].values,
})
tabla_final


,ID,Encuestador,Texto (primeras 12 palabras),Etiqueta real,Clasificación NB,Clasificación LR,Consenso
0,texto_0046,abraham,estamos en grado décimo y las actividades son más extensas y complejas...,desagradable,desagradable,desagradable,desagradable
1,texto_0481,jorge,Siento que paso mucho tiempo en la escuela y me queda poco...,desagradable,desagradable,desagradable,desagradable
2,texto_0387,jair,Son muchos además que son de un día para otro para realizar...,desagradable,desagradable,desagradable,desagradable
3,texto_0120,eberto,Paso gran parte del día en la institución y normalmente llego muy...,desagradable,desagradable,desagradable,desagradable
4,texto_0244,elvis,Pienso que el horario escolar es muy largo....,desagradable,placentero,placentero,placentero
5,texto_0241,elvis,Creo que el tiempo libre que tengo es suficiente si me organizo....,placentero,placentero,placentero,placentero
6,texto_0796,menco,"Creo que el colegio es importante, pero el tiempo en casa también...",placentero,mixto,mixto,mixto
7,texto_0098,eberto,El horario me parece adecuado porque todavía tengo tiempo para descansar y...,placentero,placentero,placentero,placentero
8,texto_0693,marlon,Bien porque pasamos mas tiempo en casa que en clase yo digo...,placentero,placentero,placentero,placentero
9,texto_0024,abraham,me siento cómodo en el colegio porque hay buen ambiente en el...,placentero,placentero,placentero,placentero


In [10]:
aciertos = (tabla_final["Etiqueta real"] == tabla_final["Consenso"]).sum()
print(f"Aciertos del consenso NB/LR sobre los {len(tabla_final)} textos de prueba: {aciertos} / {len(tabla_final)}")
print()
print("Aciertos por categoría real:")
tabla_final['acierto'] = tabla_final["Etiqueta real"] == tabla_final["Consenso"]
print(tabla_final.groupby("Etiqueta real")["acierto"].agg(['sum', 'count']))


Aciertos del consenso NB/LR sobre los 15 textos de prueba: 11 / 15

Aciertos por categoría real:
               sum  count
Etiqueta real            
desagradable     4      5
mixto            3      5
placentero       4      5


## Comparación con la versión anterior del proyecto

La versión anterior evaluaba el sistema sobre solo 6 textos de prueba (con un
corpus de entrenamiento de 193 textos, muy desbalanceado hacia
"desagradable"). Aquí se evalúa sobre 15 (5 por categoría) con un corpus de
entrenamiento 3.5 veces más grande y mucho más balanceado.

In [11]:
comparacion_versiones = pd.DataFrame({
    "Aspecto": [
        "Textos de entrenamiento",
        "Balance de categorías",
        "Accuracy SVM (Fase 2)",
        "Textos de prueba (Fase 5)",
        "Aciertos del consenso",
    ],
    "Versión anterior (1 encuestador)": [
        193, "56% / 35% / 9%", "0.667", 6, "4/6 (66.7%)",
    ],
    "Versión actual (12 encuestadores)": [
        len(sistema.textos),
        "39% / 24% / 37%",
        f"{resultados_clasificacion['SVM']['accuracy']:.3f}",
        len(tabla_final),
        f"{aciertos}/{len(tabla_final)} ({aciertos/len(tabla_final)*100:.1f}%)",
    ],
})
comparacion_versiones


,Aspecto,Versión anterior (1 encuestador),Versión actual (12 encuestadores)
0,Textos de entrenamiento,193,843
1,Balance de categorías,56% / 35% / 9%,39% / 24% / 37%
2,Accuracy SVM (Fase 2),0.667,0.799
3,Textos de prueba (Fase 5),6,15
4,Aciertos del consenso,4/6 (66.7%),11/15 (73.3%)


## Evaluación general del sistema

In [12]:
evaluacion_sistema = pd.DataFrame({
    "Componente": ["Preprocesamiento", "Clasificación", "Clustering", "Generación", "Integración"],
    "Puntuación (1-5)": [5, 4, 3, 4, 5],
    "Comentarios": [
        "Pipeline completo y reutilizable (ProcesadorNLP): limpieza, tokenización, stopwords, stemming, TF-IDF.",
        f"Accuracy ~{resultados_clasificacion['Naive Bayes']['accuracy']:.2f}-{resultados_clasificacion['SVM']['accuracy']:.2f}; subió de forma notable frente a la versión de un solo encuestador (0.64-0.67) gracias a un corpus más grande y balanceado.",
        "Silhouette bajo en todo el rango de k probado (Fase 3): los clusters reflejan más el TEMA (tiempo libre vs. tareas) que la polaridad emocional; coincidencia parcial con las categorías reales.",
        "Orden 2 genera texto notablemente más coherente que orden 1; con más de 2000 estados por categoría las generaciones son menos repetitivas que en la versión anterior.",
        "El sistema conecta las 3 técnicas end-to-end sobre datos reales de 12 fuentes nunca vistos, con resultados auditables en cada paso y una tasa de acierto de consenso más alta que antes.",
    ],
})
evaluacion_sistema


,Componente,Puntuación (1-5),Comentarios
0,Preprocesamiento,5,"Pipeline completo y reutilizable (ProcesadorNLP): limpieza, tokenización, stopwords, s..."
1,Clasificación,4,Accuracy ~0.78-0.80; subió de forma notable frente a la versión de un solo encuestador...
2,Clustering,3,Silhouette bajo en todo el rango de k probado (Fase 3): los clusters reflejan más el T...
3,Generación,4,Orden 2 genera texto notablemente más coherente que orden 1; con más de 2000 estados p...
4,Integración,5,El sistema conecta las 3 técnicas end-to-end sobre datos reales de 12 fuentes nunca vi...


## Reflexión final

**¿Cómo se complementan las tres técnicas en tu sistema?**
El preprocesamiento (Fase 1) alimenta a las otras dos: la misma
representación TF-IDF se usa tanto para clasificar (Fase 2, con
etiquetas) como para agrupar (Fase 3, sin etiquetas). El generador de
Markov (Fase 4) es independiente de TF-IDF (trabaja directamente sobre
las palabras), pero se conecta con la clasificación en la Fase 5: primero
se predice la categoría de un texto nuevo, y luego se usa el generador
Markov *de esa categoría* para producir una continuación de estilo
similar. Así, clasificación + generación forman un ciclo:
*texto -> categoría -> estilo de generación*.

**¿Qué componente fue más difícil de implementar en esta versión?**
La **reconstrucción de la Fase 0** a partir de 12 fuentes con 5 formatos de
archivo distintos, sin perder de vista la calidad: hubo que validar cada
patrón de parseo contra el 100% de los archivos de cada encuestador (no una
muestra), y luego recalibrar el lexicón de sentimientos dos veces,
verificando explícitamente que las ampliaciones de vocabulario no
introdujeran falsos positivos en textos que ya estaban etiquetados
correctamente. Fue un trabajo distinto al de la primera versión (que
alcanzaba con calibrar contra un solo estilo de escritura), pero
necesario para que el corpus reflejara la recolección real de todo el
grupo y no solo la de un compañero.

**¿Qué mejoró concretamente al pasar de 1 a 12 encuestadores?**
Tres cosas medibles: (1) el corpus de entrenamiento creció 3.5x (193 -> 674
usados en la Fase 2) sin reducir la calidad del etiquetado; (2) el balance
de categorías mejoró sustancialmente (la categoría "placentero" dejó de ser
un caso extremo de 16 ejemplos); y (3) la precisión de los tres
clasificadores subió de forma consistente (SVM: 0.667 -> 0.799, ver Fase 2).
Esto confirma que buena parte de las limitaciones reportadas en la versión
anterior (el pobre desempeño en "placentero", el fuerte desbalance) eran
consecuencia directa de usar los datos de un solo encuestador, no
limitaciones del enfoque técnico en sí.

**Si tuvieras más tiempo, ¿qué mejorarías?**
(1) Reducir el 11% de textos en "revisar_manualmente" con una revisión
humana real de esa lista (hoy solo se documenta y se excluye). (2) Afinar
el manejo de negaciones complejas en el lexicón (frases como "no considero que sea excesivo" ya se manejan, pero hay construcciones más
elaboradas que seguramente aún se escapan). (3) Probar un modelo de
generación con suavizado (backoff de orden 2 a orden 1 cuando un estado no
se ha visto) en vez de reiniciar aleatoriamente la cadena.

**¿Qué aplicaciones prácticas tendría este sistema?**
Una institución educativa podría usarlo para monitorear, a partir de
encuestas periódicas y abiertas —posiblemente recolectadas por distintas
personas, como en este proyecto—, cómo cambia el sentimiento de los
estudiantes frente a la carga académica a lo largo del año, identificar
automáticamente respuestas que requieren atención (alta proporción de
"desagradable"/"mixto" en un grupo específico) y generar resúmenes de
estilo similar al de los propios estudiantes para reportes internos. El
hecho de que el sistema siga funcionando bien al combinar datos de 12
fuentes distintas, con formatos y estilos de escritura diferentes, es una
señal razonable de que escalaría a una recolección real más grande.